<a href="https://colab.research.google.com/github/GarimaChaubey/NLP-Natural-Language-Preprocessing-/blob/main/SLM_Fine_Tuning_Full_vs_LoRA_DistilBERT_SMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DistilBERT SMS Spam Detection: Full Fine-Tuning vs LoRA

**Practical application:** classify SMS messages as **SPAM** or **HAM** (legitimate).

This notebook compares two ways to adapt the same pretrained Small Language Model:

1. **Full fine-tuning:** update all model parameters.
2. **LoRA fine-tuning:** freeze the pretrained Transformer weights and train small low-rank adapter matrices, along with the task classification head.

| Component | Choice |
|---|---|
| Foundational model | `distilbert-base-uncased` |
| Dataset | UCI SMS Spam Collection |
| Task | Binary text classification |
| Full fine-tuning | All model weights trainable |
| LoRA | PEFT adapters on attention query/value projections |
| Libraries | PyTorch, Transformers, Datasets, PEFT |

### Learning outcomes
- Prepare and inspect a real-world labelled dataset.
- Load a pretrained DistilBERT tokenizer and classifier.
- Fine-tune the model normally and with LoRA.
- Compare trainable parameters, evaluation metrics and artifact sizes.
- Save the LoRA adapter and run inference.

> **Important:** LoRA reduces the number of trainable parameters and the size of the adapter checkpoint. It does not guarantee higher accuracy or faster wall-clock training for every small model or hardware setup. Results depend on hardware, batch size and configuration.

## 1. Install dependencies

In [ ]:
%pip install -q -U transformers datasets accelerate peft scikit-learn pandas matplotlib seaborn requests

## 2. Imports and reproducibility

In [ ]:
import os, random, zipfile, io, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer, set_seed
)
from peft import LoraConfig, TaskType, get_peft_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

## 3. Load the UCI SMS Spam Collection

The official UCI archive contains tab-separated rows: a label (`ham` or `spam`) and the SMS text.

If your environment blocks downloads, download the archive from the [UCI SMS Spam Collection page](https://archive.ics.uci.edu/dataset/228/sms+spam+collection), upload the `SMSSpamCollection` file, and adapt the fallback cell.

In [ ]:
UCI_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
response = requests.get(UCI_URL, timeout=60)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
    data_file = next(n for n in zf.namelist() if n.endswith("SMSSpamCollection"))
    with zf.open(data_file) as f:
        df = pd.read_csv(
            f, sep="\\t", header=None, names=["label", "text"], encoding="utf-8"
        )

df.head()

### 3.1 Explore and clean

In [ ]:
print("Original shape:", df.shape)
print(df["label"].value_counts())
print("Missing values:\\n", df.isna().sum())

df = df.dropna(subset=["text", "label"]).drop_duplicates().reset_index(drop=True)
df["label"] = df["label"].map({"ham": 0, "spam": 1})
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

print("Cleaned shape:", df.shape)
display(df.sample(5, random_state=SEED))

In [ ]:
sns.countplot(data=df, x="label")
plt.xticks([0, 1], ["HAM (0)", "SPAM (1)"])
plt.title("SMS class distribution")
plt.show()
print(df["label"].value_counts(normalize=True))

## 4. Stratified train / validation / test split

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)
print(f"Train={len(train_df)}, Validation={len(val_df)}, Test={len(test_df)}")

## 5. Tokenizer and Hugging Face datasets

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
id2label = {0: "HAM", 1: "SPAM"}
label2id = {"HAM": 0, "SPAM": 1}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(frame):
    return Dataset.from_pandas(
        frame[["text", "label"]].reset_index(drop=True),
        preserve_index=False
    )

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize_batch, batched=True).rename_column("label", "labels")
val_ds = val_ds.map(tokenize_batch, batched=True).rename_column("label", "labels")
test_ds = test_ds.map(tokenize_batch, batched=True).rename_column("label", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(train_ds)

## 6. Shared metrics and evaluation helpers

We report spam-class precision, recall and F1 in addition to accuracy. Since the dataset is imbalanced, accuracy alone is not sufficient.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

def make_training_args(output_dir, epochs=3):
    # evaluation_strategy is supported by current Transformers releases.
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=epochs,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=50,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED
    )

## 7. Experiment A — Full fine-tuning

All parameters in DistilBERT, including the classification head, are trainable. This is the standard fine-tuning baseline.

In [ ]:
full_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
)

full_total = sum(p.numel() for p in full_model.parameters())
full_trainable = sum(p.numel() for p in full_model.parameters() if p.requires_grad)
print(f"Full model parameters: {full_total:,}")
print(f"Trainable parameters: {full_trainable:,} ({100*full_trainable/full_total:.2f}%)")

full_trainer = Trainer(
    model=full_model,
    args=make_training_args("./results_full_finetuning"),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

start_full = time.time()
full_train_result = full_trainer.train()
full_train_seconds = time.time() - start_full
print(f"Full fine-tuning wall time: {full_train_seconds/60:.2f} minutes")

### 7.1 Evaluate full fine-tuning on the held-out test set

In [ ]:
full_test_metrics = full_trainer.evaluate(test_ds)
print(full_test_metrics)

full_pred_output = full_trainer.predict(test_ds)
y_true = full_pred_output.label_ids
full_preds = np.argmax(full_pred_output.predictions, axis=1)

print(classification_report(
    y_true, full_preds, target_names=["HAM", "SPAM"], digits=4, zero_division=0
))

In [ ]:
sns.heatmap(
    confusion_matrix(y_true, full_preds), annot=True, fmt="d", cmap="Blues",
    xticklabels=["HAM", "SPAM"], yticklabels=["HAM", "SPAM"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Full fine-tuning — test confusion matrix")
plt.show()

## 8. Experiment B — LoRA fine-tuning

LoRA injects trainable low-rank matrices into selected linear layers. The pretrained backbone weights are frozen; only the LoRA adapter weights and task head are trained.

**DistilBERT attention layer names:** `q_lin` and `v_lin`.

We set `modules_to_save=["pre_classifier", "classifier"]` so the task-specific classification head is also trained and saved. This is important: training only attention adapters while leaving a randomly initialized classifier frozen would be an invalid comparison.

In [ ]:
lora_base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    modules_to_save=["pre_classifier", "classifier"],
    bias="none"
)

lora_model = get_peft_model(lora_base_model, lora_config)
lora_model.print_trainable_parameters()

### 8.1 Train the LoRA adapters

In [ ]:
lora_trainer = Trainer(
    model=lora_model,
    args=make_training_args("./results_lora_finetuning"),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

start_lora = time.time()
lora_train_result = lora_trainer.train()
lora_train_seconds = time.time() - start_lora
print(f"LoRA fine-tuning wall time: {lora_train_seconds/60:.2f} minutes")

### 8.2 Evaluate LoRA on the same held-out test set

In [ ]:
lora_test_metrics = lora_trainer.evaluate(test_ds)
print(lora_test_metrics)

lora_pred_output = lora_trainer.predict(test_ds)
lora_preds = np.argmax(lora_pred_output.predictions, axis=1)

print(classification_report(
    y_true, lora_preds, target_names=["HAM", "SPAM"], digits=4, zero_division=0
))

In [ ]:
sns.heatmap(
    confusion_matrix(y_true, lora_preds), annot=True, fmt="d", cmap="Greens",
    xticklabels=["HAM", "SPAM"], yticklabels=["HAM", "SPAM"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("LoRA fine-tuning — test confusion matrix")
plt.show()

## 9. Compare Full Fine-Tuning vs LoRA

The trainable-parameter count is measured directly from the models. Both approaches use the same dataset split and evaluation metrics. Training time is a measurement from this particular runtime and should not be treated as a universal benchmark.

In [ ]:
lora_total = sum(p.numel() for p in lora_model.parameters())
lora_trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)

comparison = pd.DataFrame([
    {
        "Approach": "Full fine-tuning",
        "Total parameters": full_total,
        "Trainable parameters": full_trainable,
        "Trainable (%)": 100 * full_trainable / full_total,
        "Test accuracy": full_test_metrics.get("eval_accuracy"),
        "Spam precision": full_test_metrics.get("eval_precision"),
        "Spam recall": full_test_metrics.get("eval_recall"),
        "Spam F1": full_test_metrics.get("eval_f1"),
        "Training time (min)": full_train_seconds / 60
    },
    {
        "Approach": "LoRA",
        "Total parameters": lora_total,
        "Trainable parameters": lora_trainable,
        "Trainable (%)": 100 * lora_trainable / lora_total,
        "Test accuracy": lora_test_metrics.get("eval_accuracy"),
        "Spam precision": lora_test_metrics.get("eval_precision"),
        "Spam recall": lora_test_metrics.get("eval_recall"),
        "Spam F1": lora_test_metrics.get("eval_f1"),
        "Training time (min)": lora_train_seconds / 60
    }
])
display(comparison.round(4))

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=comparison, x="Approach", y="Trainable (%)")
plt.title("Percentage of trainable parameters")
plt.ylabel("Trainable parameters (%)")
plt.show()

## 10. Save both model artifacts

- Full fine-tuning saves the complete model and tokenizer.
- LoRA saves the adapter weights/configuration plus the tokenizer. The original base model is still needed when loading the adapter, unless you merge the adapter weights into the base model.

In [ ]:
FULL_SAVE_DIR = "./sms-spam-distilbert-full"
LORA_SAVE_DIR = "./sms-spam-distilbert-lora"

full_trainer.save_model(FULL_SAVE_DIR)
tokenizer.save_pretrained(FULL_SAVE_DIR)

lora_trainer.model.save_pretrained(LORA_SAVE_DIR)
tokenizer.save_pretrained(LORA_SAVE_DIR)

print("Full model saved to:", FULL_SAVE_DIR)
print("LoRA adapter saved to:", LORA_SAVE_DIR)

### 10.1 Compare artifact sizes

In [ ]:
def directory_size_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for name in files:
            total += os.path.getsize(os.path.join(root, name))
    return total / (1024 ** 2)

print(f"Full model artifact: {directory_size_mb(FULL_SAVE_DIR):.2f} MB")
print(f"LoRA adapter artifact: {directory_size_mb(LORA_SAVE_DIR):.2f} MB")

## 11. Load the saved LoRA adapter and perform inference

The PEFT model is reconstructed by loading the same base model and attaching the saved adapter.

In [ ]:
from peft import PeftModel

inference_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
)
lora_inference_model = PeftModel.from_pretrained(inference_base, LORA_SAVE_DIR)
lora_inference_model.eval()

def predict_sms_lora(message):
    inputs = tokenizer(
        message, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    lora_inference_model.to(DEVICE)
    with torch.no_grad():
        logits = lora_inference_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
    pred_id = int(torch.argmax(probs).item())
    return {
        "message": message,
        "prediction": id2label[pred_id],
        "confidence": round(float(probs[pred_id].item()), 4),
        "ham_probability": round(float(probs[0].item()), 4),
        "spam_probability": round(float(probs[1].item()), 4)
    }

sample_messages = [
    "Congratulations! You have won a free prize. Call now to claim.",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT! Your account has been selected for a cash reward. Click here."
]
for message in sample_messages:
    print(predict_sms_lora(message))

## 12. Try your own message

In [ ]:
my_sms = "Your package delivery is delayed. Please check the official app."
predict_sms_lora(my_sms)

## 13. Interpretation, caveats and further experiments

### What to look for
- **Trainable parameters:** LoRA should have substantially fewer trainable parameters than full fine-tuning.
- **Test F1:** compare spam-class F1, precision and recall; LoRA may be close to full fine-tuning, but it is not guaranteed.
- **Artifact size:** a LoRA adapter is usually much smaller than a complete model checkpoint.
- **Training time:** measure it on the same hardware; LoRA can reduce optimizer state and gradient memory, but may not always run faster.

### Practical limitations
- This is a relatively small, older SMS dataset. Real scam messages evolve, so external validation on recent data is important.
- A false positive can hide a legitimate message. Choose a spam threshold based on the real cost of false positives and false negatives.
- Confidence scores are not calibrated probabilities by default.
- Avoid sending private SMS content to third-party services without appropriate consent and safeguards.

### Further experiments
1. Try LoRA ranks `r=4`, `r=8`, and `r=16`.
2. Change `lora_alpha` and compare validation F1.
3. Tune a classification threshold using validation predictions only.
4. Compare with `bert-base-uncased`.
5. Test on a newer independent SMS dataset.

## Conclusion

This notebook demonstrates two practical model-adaptation strategies for a compact pretrained Transformer. Full fine-tuning updates the entire model, while LoRA trains a small set of adapter parameters and the classification head. The side-by-side evaluation helps quantify the trade-offs in trainable parameters, quality, runtime and storage.